In [1]:
import time
import serial
import datetime as dt
import numpy as np
from pathlib import Path

In [2]:
savepath = Path("C:/Users/IPMU/Desktop/ipmu_DAQ/GaussMeter")
savepath.mkdir(parents=True, exist_ok=True)

filename = savepath / f"{dt.datetime.now().strftime('%Y%m%d%H%M%S')}_GaussMeter.csv"
print(filename)

columnname = ['Abstime', 'Reltime','Intensity [mT]',]

try:
    with open(filename, mode="a") as f:
        print(*columnname, sep=", ", file=f)
except:
    pass

C:\Users\IPMU\Desktop\ipmu_DAQ\GaussMeter\20260612150843_GaussMeter.csv


In [3]:
PORT = "COM9"
BAUDRATE = 57600

ser = serial.Serial(
    port=PORT,
    baudrate=BAUDRATE,
    bytesize=serial.SEVENBITS,     # 7 data bits
    parity=serial.PARITY_ODD,      # odd parity
    stopbits=serial.STOPBITS_ONE,  # 1 stop bit
    timeout=1.0,                   # Timeout
    write_timeout=1.0,
)

def send_command(ser, command, do_print=True):
    cmd = command.strip()
    msg = cmd + "\n"

    ser.write(msg.encode("ascii"))
    ser.flush()

    response = None
    if cmd.endswith("?"):
        response = ser.read_until(b"\n").decode("ascii", errors="replace").strip()
        if do_print:
            print(response)

    time.sleep(0.03)
    return response

In [4]:
send_command(ser, "*IDN?", do_print=False)

'LSCI,MODEL425,LSA19DO,1.3'

In [5]:
# set parameters
send_command(ser, "RANGE 2")
send_command(ser, "UNIT 2")
send_command(ser, "RDGMODE 1,1,2")
rx = send_command(ser, "RDGFIELD?")
print(rx)

-033.580E-03
-033.580E-03


In [6]:
t_init = time.time()
SAMPRINGRATE = 0.1

try:
    while True:
        t_abs = dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        t_rel = time.time() - t_init
        t_arr = [t_abs, t_rel]

        rx = send_command(ser, "RDGFIELD?", do_print=False)
        v = float(rx)
        B_mT = v*1000

        arr = np.hstack([t_arr, B_mT])

        print(arr)
        time.sleep(SAMPRINGRATE)

        try:
            with open(filename, mode="a") as f:
                print(*arr, sep=", ", file=f)

        except FileNotFoundError:
            print("File Not Found Error... (´；Д；｀)")
            filename = savepath / f"{dt.datetime.now().strftime('%Y%m%d%H%M%S')}_Pressure.csv"
            with open(filename, mode="a") as f:
                print(*columnname, sep=", ", file=f)
                print(*arr, sep=", ", file=f)
finally:
    ser.close()


['2026-06-12 15:08:43' '0.0' '-33.54']
['2026-06-12 15:08:44' '0.20692062377929688' '-33.529999999999994']
['2026-06-12 15:08:44' '0.3508949279785156' '-33.54']
['2026-06-12 15:08:44' '0.5127580165863037' '-33.54']
['2026-06-12 15:08:44' '0.6822710037231445' '-33.54']
['2026-06-12 15:08:44' '0.8572347164154053' '-33.54']
['2026-06-12 15:08:44' '1.0047459602355957' '-33.54']
['2026-06-12 15:08:45' '1.155975580215454' '-33.54']
['2026-06-12 15:08:45' '1.3026049137115479' '-33.54']
['2026-06-12 15:08:45' '1.4457135200500488' '-33.54']
['2026-06-12 15:08:45' '1.6085827350616455' '-33.550000000000004']
['2026-06-12 15:08:45' '1.7534916400909424' '-33.54']
['2026-06-12 15:08:45' '1.901153326034546' '-33.550000000000004']
['2026-06-12 15:08:45' '2.0524330139160156' '-33.54']
['2026-06-12 15:08:46' '2.205127239227295' '-33.54']
['2026-06-12 15:08:46' '2.357285499572754' '-33.54']
['2026-06-12 15:08:46' '2.5212528705596924' '-33.54']
['2026-06-12 15:08:46' '2.6846420764923096' '-33.54']
['2026-

KeyboardInterrupt: 